# C1.9 · Machine-speed containment and fleet revocation

**Function C — Agentic Evaluation and Red Teaming → One Red-Team Lifecycle, End to End**

Builds on **[C1.8 · Defensive deception and threshold failures](https://spbreed.github.io/cyber-commons/lessons/C1.8.html)**.

| | |
|---|---|
| Tools used | SPIFFE |

## What this lesson is

**What it covers.** Zero-trust runtime gatekeepers and dynamic token revocation that isolate a compromised fleet in one action, at machine speed.

**Why a security engineer needs it.** A fleet completes thousands of actions inside one approval cycle, so containment must be pre-authorised. The property that decides whether it worked is revocation versus termination — terminated agents with valid tokens keep their persistence.

| | |
|---|---|
| **Day 0 — why** | A fleet completes thousands of actions inside one human approval cycle, so containment is pre-authorised or it is too late. |
| **Day 1 — how** | Build a zero-trust gatekeeper and a revocation path that isolates the whole fleet in one action. |
| **Day 2 — measure** | What the credentials can still do after the agents stop. Termination without revocation moves the incident rather than ending it. |

## 1 · The hook

An agent fleet completes thousands of actions inside one human approval cycle, so containment is pre-authorised or it is too late. Terminating the agents while their tokens stay valid moves the incident rather than ending it.

> **At CyberTravels.** The fleet is CyberTravels' four agents, and the detail that matters is R9: third-party access ended only when the third party revoked its keys, not when the agents were stopped.

## 2 · The framework

```
   an agent at machine speed vs a human approval cycle

   approval cycle: 8 min  ->  ~2,400 further agent actions
   automated stop:          ->  ~60

   one revocation, at the gateway, reaching the whole fleet
     terminate processes   -> tokens still valid -> persistence remains
     REVOKE credentials    -> persistence ends       <- the property that counts
```

Containment against a swarm is a race the defender starts behind. An agent fleet
operating at machine speed completes thousands of actions inside a human
approval cycle, so the containment has to be **pre-authorised and automated**:
zero-trust runtime gatekeepers, and dynamic token revocation that isolates the
fleet in one action.

The detail that decides whether containment worked is revocation versus
termination. Terminating agents while their bearer tokens stay valid leaves the
persistence in place and moves the incident rather than ending it — which is the
Moltbook lesson, 770,000 agents behind one missing policy.

## 3 · The procedure, as a skill

The skill exercises a fleet kill switch against CyberTravels' agents and checks the property that matters — that revoked credentials, not just terminated processes, are what ends the persistence.

### The skill — [`skills/response/fleet-kill-switch-test/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/fleet-kill-switch-test/SKILL.md)

```yaml
name: fleet-kill-switch-test
description: >-
  Test a fleet-wide stop for what it leaves behind — valid tokens, destroyed
  evidence, and the runs that should have been preserved — and measure how long
  the whole thing takes. Use before relying on a kill switch, and after
  designing one.
allowed-tools: Read, Grep, Glob
```

# Terminating the agents leaves every token valid

A fleet kill switch usually terminates workloads. Termination does not revoke
credentials, so every token the fleet held stays valid until it expires — up to
seventy-two hours of an attacker being able to act as agents that no longer
exist. And a kill that does not preserve first destroys the evidence for the
incident that triggered it.

## When to use this

Before a kill switch is relied on, after it is built, and once a year as a
rehearsal.

## Procedure

**1 — Terminate only, and count what stays valid.** Tokens, sessions,
outstanding delegated grants. This is the baseline failure and it is invisible
in a design review.

**2 — Terminate and revoke together, and re-count.** Zero is the target. If
revocation is a separate runbook step performed by a different team, it will not
happen at speed.

**3 — Check evidence preservation.** Kill with and without preservation, then
attempt the reconstruction. A kill switch that destroys the run records is a
containment that ends the investigation.

**4 — Test selectivity.** Stop one agent, one class, the whole fleet. A switch
with only the last setting will not be used until it is far too late, which is
the same as not having one.

**5 — Measure the time.** Decision to last agent stopped and last token revoked.
Set a target and report against it; an untimed rehearsal is a demonstration.

## Example

**Input** — the fixture committed at the top of [`scripts/fleet_kill_switch_test.py`](scripts/fleet_kill_switch_test.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
terminate only        : 8 agents stopped, 8 tokens still valid for up to 72h
terminate and revoke  : 8 agents stopped, 0 tokens still valid

In the source incident, third-party access ended when the third party
revoked its keys - not when the agents stopped. Stopping the process is
the visible half of containment and the smaller one.
terminate first   terminate -> revoke credentials                           reconstructable=False
preserve first    snapshot state and transcripts -> terminate -> revoke credentialsreconstructable=True
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "fleet": {"agents": 0, "tokens": 0},
  "terminate_only": {"tokens_valid_after": 0, "max_validity_hours": 0},
  "terminate_and_revoke": {"tokens_valid_after": 0},
  "preservation": {"preserved": true, "reconstructable": true},
  "selectivity": [{"scope": "one|class|fleet", "supported": true}],
  "timing": {"target_minutes": 0.0, "measured_minutes": 0.0}
}
```

## Failure modes

- **Terminating without revoking.** The credentials outlive the workloads.
- **Killing before preserving.** The incident becomes unreconstructable.
- **A fleet-only switch.** Nobody uses it until the whole estate is on fire.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/fleet-kill-switch-test/scripts/fleet_kill_switch_test.py
SCRIPT = "skills/response/fleet-kill-switch-test/scripts/fleet_kill_switch_test.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.5 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The kill switch selects the whole fleet in one action and the run shows persistence surviving termination but not revocation, so the control is judged on what the tokens can still do afterwards.

## Your turn

Time how long it takes to revoke every credential one class of agent holds. If the answer is 'we would terminate the processes', you have not tested containment, only restart.

---

**Next → [C1.10 · Forensic replay and control architecture](https://spbreed.github.io/cyber-commons/lessons/C1.10.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*